# 03e_import_2wikimultihop.ipynb

Импорт и первичная нормализация **2WikiMultiHopQA** как следующего внешнего слоя для benchmark-а.

Что делает ноутбук:
- скачивает официальный mirror датасета **`xanhho/2WikiMultihopQA`** с Hugging Face через `snapshot_download`;
- находит файлы train/dev/test и загружает их;
- приводит записи к единой схеме;
- считает полезные признаки сложности:
  - `question_type`
  - `num_supporting_facts`
  - `num_context_docs`
  - `num_evidences`
  - `num_answer_aliases`
  - `bridge_depth_estimate`
- выделяет:
  - `all_normalized`
  - `strong_multihop_candidates`
  - `translation_minimal`
  - `manual_review_sheet`
- экспортирует в `jsonl / json / csv`.

Важно:
- здесь **нет перевода на русский**;
- **нет dedup**;
- **нет финального санитичека**;
- это именно ноутбук для **import + normalize + initial filtering**.

In [1]:
# =========================
# УСТАНОВКА ЗАВИСИМОСТЕЙ
# =========================
%pip install -q pandas pyarrow requests huggingface_hub datasets tqdm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# =========================
# ИМПОРТЫ И КОНФИГ
# =========================
from __future__ import annotations

import csv
import json
import hashlib
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

import pandas as pd
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts_2wikimultihop_stage1"
RAW_DIR = ARTIFACTS_DIR / "_cache_raw"
JSONL_DIR = ARTIFACTS_DIR / "jsonl"
CSV_DIR = ARTIFACTS_DIR / "csv"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

for p in [ARTIFACTS_DIR, RAW_DIR, JSONL_DIR, CSV_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

HF_DATASET_REPO = "xanhho/2WikiMultihopQA"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("ARTIFACTS_DIR =", ARTIFACTS_DIR)

PROJECT_ROOT = /Users/matvey/Desktop/2wikimultihop
ARTIFACTS_DIR = /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =========================
# БАЗОВЫЕ УТИЛИТЫ
# =========================
def normalize_spaces(text: Optional[str]) -> str:
    if text is None:
        return ""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def stable_hash(text: str, n: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:n]

def unique_preserve_order(items: Iterable[Any]) -> List[Any]:
    seen = set()
    out = []
    for x in items:
        key = json.dumps(x, ensure_ascii=False, sort_keys=True) if isinstance(x, (dict, list)) else str(x)
        if key not in seen:
            seen.add(key)
            out.append(x)
    return out

def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(rows: List[Dict[str, Any]], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\\n")

def write_json(rows: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

def to_list(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]

In [4]:
# =========================
# СКАЧИВАНИЕ SNAPSHOT С HF
# =========================
# Почему не load_dataset:
# у ряда mirror-репозиториев удобнее и надёжнее скачать snapshot целиком,
# а дальше самому найти raw train/dev/test.

snapshot_path = snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    local_dir=RAW_DIR / "hf_snapshot",
    local_dir_use_symlinks=False,
)

SNAPSHOT_DIR = Path(snapshot_path)
print("SNAPSHOT_DIR =", SNAPSHOT_DIR)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 9 files: 100%|██████████| 9/9 [00:59<00:00,  6.65s/it]

SNAPSHOT_DIR = /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1/_cache_raw/hf_snapshot


In [5]:
# =========================
# ПОИСК RAW-ФАЙЛОВ TRAIN / DEV / TEST
# =========================
def discover_split_files(root: Path) -> Dict[str, Path]:
    split_candidates: Dict[str, List[Path]] = defaultdict(list)

    for p in root.rglob("*"):
        if not p.is_file():
            continue
        name = p.name.lower()
        if p.suffix.lower() not in {".json", ".jsonl", ".parquet"}:
            continue

        if re.search(r"(^|[_\-])(train)([_\-\.]|$)", name):
            split_candidates["train"].append(p)
        elif re.search(r"(^|[_\-])(dev|valid|validation)([_\-\.]|$)", name):
            split_candidates["dev"].append(p)
        elif re.search(r"(^|[_\-])(test)([_\-\.]|$)", name):
            split_candidates["test"].append(p)

    chosen: Dict[str, Path] = {}
    for split, files in split_candidates.items():
        # Предпочитаем parquet, если он есть: у mirror 2WikiMultihopQA на HF лежат train/dev/test.parquet.
        files = sorted(
            files,
            key=lambda x: (
                0 if x.suffix.lower() == ".parquet" else 1,
                len(x.parts),
                len(str(x)),
            ),
        )
        chosen[split] = files[0]

    return chosen

split_files = discover_split_files(SNAPSHOT_DIR)
print("[discovered split files]")
for k, v in split_files.items():
    print(f"  {k}: {v.relative_to(SNAPSHOT_DIR)}")

if not split_files:
    raise RuntimeError("Не удалось найти raw train/dev/test json/jsonl/parquet-файлы в snapshot 2WikiMultiHopQA.")


[discovered split files]
  train: train.parquet
  test: test.parquet
  dev: dev.parquet


In [6]:
# =========================
# ЗАГРУЗКА RAW SPLIT-ОВ
# =========================
def load_rows(path: Path) -> List[Dict[str, Any]]:
    suffix = path.suffix.lower()

    if suffix == ".jsonl":
        return read_jsonl(path)

    if suffix == ".parquet":
        df = pd.read_parquet(path)
        return df.to_dict(orient="records")

    obj = read_json(path)

    if isinstance(obj, list):
        return obj

    if isinstance(obj, dict):
        for key in ["data", "examples", "rows", "instances"]:
            if key in obj and isinstance(obj[key], list):
                return obj[key]

    raise ValueError(f"Неожиданный формат файла: {path}")

raw_by_split: Dict[str, List[Dict[str, Any]]] = {}
for split, path in split_files.items():
    rows = load_rows(path)
    raw_by_split[split] = rows
    print(f"[load] {split}: {len(rows):,} rows from {path.name}")


[load] train: 167,454 rows from train.parquet
[load] test: 12,576 rows from test.parquet
[load] dev: 12,576 rows from dev.parquet


In [7]:
# =========================
# ИНСПЕКЦИЯ СХЕМЫ RAW 2WIKI
# =========================
for split, rows in raw_by_split.items():
    if rows:
        sample = rows[0]
        print("=" * 100)
        print("split =", split)
        print("keys =", sorted(sample.keys()))
        print("sample question =", sample.get("question"))
        print("sample type =", sample.get("type"))
        print("sample answer =", sample.get("answer"))
        print("sample supporting_facts[:2] =", str(sample.get("supporting_facts", []))[:300])
        print("sample evidences[:2] =", str(sample.get("evidences", []))[:300])
        break

split = train
keys = ['_id', 'answer', 'context', 'evidences', 'question', 'supporting_facts', 'type']
sample question = Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
sample type = bridge_comparison
sample answer = no
sample supporting_facts[:2] = [["Move (1970 film)", 0], ["M\u00e9diterran\u00e9e (1963 film)", 0], ["Stuart Rosenberg", 0], ["Jean-Daniel Pollet", 0]]
sample evidences[:2] = [["Move (1970 film)", "director", "Stuart Rosenberg"], ["M\u00e9diterran\u00e9e (1963 film)", "director", "Jean-Daniel Pollet"], ["Stuart Rosenberg", "country of citizenship", "American"], ["Jean-Daniel Pollet", "country of citizenship", "French"]]


In [8]:
# =========================
# НОРМАЛИЗАЦИЯ ANSWER / EVIDENCE / CONTEXT
# =========================
def extract_answer_texts(answer: Any) -> List[str]:
    out: List[str] = []

    if answer is None:
        return out

    if isinstance(answer, str):
        answer = normalize_spaces(answer)
        if answer:
            out.append(answer)
        return unique_preserve_order(out)

    if isinstance(answer, (int, float, bool)):
        out.append(str(answer))
        return unique_preserve_order(out)

    if isinstance(answer, list):
        for item in answer:
            out.extend(extract_answer_texts(item))
        return unique_preserve_order(out)

    if isinstance(answer, dict):
        for k in ["text", "answer", "label", "name", "value", "answers"]:
            if k in answer:
                out.extend(extract_answer_texts(answer[k]))
        return unique_preserve_order(out)

    return out

def extract_supporting_facts(x: Any) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    if not isinstance(x, list):
        return out
    for item in x:
        if isinstance(item, list) and len(item) >= 2:
            out.append({"title": str(item[0]), "sent_id": item[1]})
        elif isinstance(item, dict):
            out.append({
                "title": item.get("title"),
                "sent_id": item.get("sent_id", item.get("sentence_id"))
            })
    return out

def extract_context_docs(x: Any) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    if not isinstance(x, list):
        return out
    for item in x:
        if isinstance(item, list) and len(item) >= 2:
            title = item[0]
            sentences = item[1] if isinstance(item[1], list) else []
            out.append({
                "title": title,
                "sentences": [normalize_spaces(s) for s in sentences if normalize_spaces(s)]
            })
        elif isinstance(item, dict):
            title = item.get("title")
            sentences = to_list(item.get("sentences"))
            out.append({
                "title": title,
                "sentences": [normalize_spaces(s) for s in sentences if normalize_spaces(s)]
            })
    return out

def extract_evidences(x: Any) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    if not isinstance(x, list):
        return out
    for item in x:
        if isinstance(item, list) and len(item) >= 3:
            out.append({
                "subject": item[0],
                "relation": item[1],
                "object": item[2],
            })
        elif isinstance(item, dict):
            out.append({
                "subject": item.get("subject"),
                "relation": item.get("relation"),
                "object": item.get("object"),
            })
    return out

def estimate_bridge_depth(question: str) -> int:
    q = " " + normalize_spaces(question).lower() + " "
    score = 0
    for token in [" who ", " whose ", " where ", " when ", " that ", " which ", " of the ", " in the "]:
        score += q.count(token)
    return score

In [9]:
# =========================
# РАЗМЕТКА ТИПА ОТВЕТА И СИЛЫ ВОПРОСА
# =========================
def infer_answer_mode(answer_texts: List[str], raw_answer: Any, q_type: str) -> str:
    q_type = (q_type or "").lower()
    if isinstance(raw_answer, bool):
        return "boolean"
    if q_type in {"comparison", "bridge_comparison"} and len(answer_texts) <= 1:
        return "single"
    if len(answer_texts) == 0:
        return "unknown"
    if len(answer_texts) == 1:
        txt = answer_texts[0].strip().lower()
        if re.fullmatch(r"-?\d+(\.\d+)?", txt):
            return "count_or_number"
        return "single"
    return "list_like"

def shallow_bridge_flag(question: str, q_type: str, num_evidences: int, num_supporting_facts: int) -> bool:
    q = normalize_spaces(question).lower()
    patterns = [
        r"^where was .+ born\\??$",
        r"^who (wrote|directed|produced|composed) .+\\??$",
        r"^where did .+ go to university\\??$",
        r"^what (genre|platform|nationality|birthplace) .+\\??$",
    ]
    if any(re.match(p, q) for p in patterns):
        return True
    if q_type in {"compositional", "inference"} and num_evidences <= 2 and num_supporting_facts <= 2:
        return True
    return False

def strength_bucket(q_type: str, answer_mode: str, num_evidences: int, num_supporting_facts: int, question: str) -> str:
    q_type = (q_type or "").lower()
    score = 0
    if q_type in {"bridge_comparison", "comparison"}:
        score += 2
    if q_type in {"compositional", "inference"}:
        score += 1
    if answer_mode == "list_like":
        score += 2
    if answer_mode == "count_or_number":
        score += 1
    if num_evidences >= 3:
        score += 2
    if num_supporting_facts >= 3:
        score += 1
    if estimate_bridge_depth(question) >= 3:
        score += 1

    if score >= 5:
        return "strong"
    if score >= 3:
        return "medium"
    return "weak"

In [10]:
# =========================
# НОРМАЛИЗАЦИЯ ВСЕХ SPLIT-ОВ
# =========================
normalized_rows: List[Dict[str, Any]] = []

for split, rows in raw_by_split.items():
    for raw in tqdm(rows, desc=f"normalize:{split}"):
        benchmark_id = raw.get("_id") or raw.get("id") or stable_hash(
            f"{split}|{raw.get('question', '')}|{json.dumps(raw, ensure_ascii=False, sort_keys=True)}"
        )
        question = normalize_spaces(raw.get("question"))
        q_type = normalize_spaces(raw.get("type")).lower()

        supporting_facts = extract_supporting_facts(raw.get("supporting_facts"))
        context_docs = extract_context_docs(raw.get("context"))
        evidences = extract_evidences(raw.get("evidences"))
        answer_texts = extract_answer_texts(raw.get("answer"))
        answer_mode = infer_answer_mode(answer_texts, raw.get("answer"), q_type)

        row = {
            "benchmark_id": f"2wiki_{benchmark_id}",
            "source_dataset": "2wikimultihopqa",
            "source_split": split,
            "question_en_original": question,
            "question_ru": None,
            "reasoning_family": q_type,
            "question_type": q_type,
            "answer_mode": answer_mode,
            "gold_answers": answer_texts,
            "gold_qids": [],
            "num_answers": len(answer_texts),
            "supporting_facts": supporting_facts,
            "num_supporting_facts": len(supporting_facts),
            "context_docs": context_docs,
            "num_context_docs": len(context_docs),
            "evidences": evidences,
            "num_evidences": len(evidences),
            "bridge_depth_estimate": estimate_bridge_depth(question),
            "shallow_bridge_flag": shallow_bridge_flag(question, q_type, len(evidences), len(supporting_facts)),
            "strength_bucket": strength_bucket(q_type, answer_mode, len(evidences), len(supporting_facts), question),
            "raw_answer": raw.get("answer"),
            "raw_type": raw.get("type"),
            "translation_ready_text": question,
            "needs_translation": True,
            "needs_rephrase": True,
        }
        normalized_rows.append(row)

print("[normalize] normalized rows:", len(normalized_rows))

normalize:dev: 100%|██████████| 12576/12576 [00:00<00:00, 51470.93it/s]

[normalize] normalized rows: 192606


In [11]:
# =========================
# DATAFRAME + QUICK SUMMARY
# =========================
df = pd.DataFrame(normalized_rows)

summary = {
    "all_rows": int(len(df)),
    "by_split": df["source_split"].value_counts(dropna=False).to_dict(),
    "by_question_type": df["question_type"].value_counts(dropna=False).to_dict(),
    "by_answer_mode": df["answer_mode"].value_counts(dropna=False).to_dict(),
    "by_strength_bucket": df["strength_bucket"].value_counts(dropna=False).to_dict(),
    "num_shallow_bridge_true": int(df["shallow_bridge_flag"].sum()),
}

summary

{'all_rows': 192606,
 'by_split': {'train': 167454, 'test': 12576, 'dev': 12576},
 'by_question_type': {'compositional': 86979,
  'comparison': 57989,
  'bridge_comparison': 40160,
  'inference': 7478},
 'by_answer_mode': {'single': 185212, 'unknown': 6812, 'count_or_number': 582},
 'by_strength_bucket': {'weak': 191026, 'medium': 1580},
 'num_shallow_bridge_true': 94457}

In [12]:
# =========================
# ФИЛЬТР "СИЛЬНЫЕ MULTIHOP КАНДИДАТЫ"
# =========================
preferred_types = {"comparison", "bridge_comparison", "compositional", "inference"}

cand_df = df.copy()
cand_df = cand_df[cand_df["question_type"].isin(preferred_types)]
cand_df = cand_df[cand_df["strength_bucket"].isin({"strong", "medium"})]
cand_df = cand_df[~cand_df["shallow_bridge_flag"]]

cand_df["reconstructability_score"] = (
    cand_df["num_evidences"].clip(upper=5) * 2
    + cand_df["num_supporting_facts"].clip(upper=5)
    + (cand_df["answer_mode"] == "list_like").astype(int) * 3
    + (cand_df["question_type"].isin(["comparison", "bridge_comparison"])).astype(int) * 2
    + (cand_df["strength_bucket"] == "strong").astype(int) * 2
    + cand_df["bridge_depth_estimate"].clip(upper=5)
)

cand_df = cand_df.sort_values(
    ["reconstructability_score", "num_evidences", "num_supporting_facts", "num_answers"],
    ascending=[False, False, False, False]
).reset_index(drop=True)

print("strong_multihop_candidates:", len(cand_df))
cand_df[["benchmark_id", "question_en_original", "question_type", "answer_mode", "num_answers", "num_evidences", "num_supporting_facts", "reconstructability_score"]].head(20)

strong_multihop_candidates: 1579


,benchmark_id,question_en_original,question_type,answer_mode,num_answers,num_evidences,num_supporting_facts,reconstructability_score
0,2wiki_e07552fc08a111ebbd7aac1f6bf848b6,"Which film has the director who is older, The ...",bridge_comparison,single,1,0,0,7
1,2wiki_414beed608bf11ebbd8aac1f6bf848b6,"Which film has the director who died first, Wh...",bridge_comparison,single,1,0,0,7
2,2wiki_414beed408bf11ebbd8aac1f6bf848b6,Which film has the director who is older than ...,bridge_comparison,single,1,0,0,7
3,2wiki_9e841e45087611ebbd67ac1f6bf848b6,"Which film whose director is younger, Time To ...",bridge_comparison,single,1,0,0,6
4,2wiki_1650b5a6086111ebbd5dac1f6bf848b6,Which film has the director who was born earli...,bridge_comparison,single,1,0,0,6
5,2wiki_2a221b8808d411ebbd96ac1f6bf848b6,Which film has the director who was born earli...,bridge_comparison,single,1,0,0,6
6,2wiki_023b608f087a11ebbd67ac1f6bf848b6,"Which film has the director who died first, Da...",bridge_comparison,single,1,0,0,6
7,2wiki_c8d52ef108c011ebbd8aac1f6bf848b6,Which film has the director who was born earli...,bridge_comparison,single,1,0,0,6
8,2wiki_bfb6685c088911ebbd6eac1f6bf848b6,"Which film has the director who is older, In T...",bridge_comparison,single,1,0,0,6
9,2wiki_64e077b608b011ebbd84ac1f6bf848b6,Which film has the director who is older than ...,bridge_comparison,single,1,0,0,6


In [13]:
# =========================
# TRANSLATION MINIMAL + MANUAL REVIEW SHEET
# =========================
translation_minimal_cols = [
    "benchmark_id",
    "source_dataset",
    "source_split",
    "question_en_original",
    "question_type",
    "reasoning_family",
    "answer_mode",
    "num_answers",
    "num_evidences",
    "num_supporting_facts",
    "reconstructability_score",
    "translation_ready_text",
    "needs_translation",
    "needs_rephrase",
]

translation_minimal_df = cand_df[translation_minimal_cols].copy()

manual_review_df = cand_df[[
    "benchmark_id",
    "question_en_original",
    "question_type",
    "reasoning_family",
    "answer_mode",
    "num_answers",
    "num_evidences",
    "num_supporting_facts",
    "bridge_depth_estimate",
    "strength_bucket",
    "reconstructability_score",
]].copy()

manual_review_df["manual_decision"] = ""
manual_review_df["manual_bucket"] = ""
manual_review_df["notes"] = ""

print("translation_minimal:", translation_minimal_df.shape)
print("manual_review_sheet:", manual_review_df.shape)
manual_review_df.head(15)

translation_minimal: (1579, 14)
manual_review_sheet: (1579, 14)


,benchmark_id,question_en_original,question_type,reasoning_family,answer_mode,num_answers,num_evidences,num_supporting_facts,bridge_depth_estimate,strength_bucket,reconstructability_score,manual_decision,manual_bucket,notes
0,2wiki_e07552fc08a111ebbd7aac1f6bf848b6,"Which film has the director who is older, The ...",bridge_comparison,bridge_comparison,single,1,0,0,5,medium,7,,,
1,2wiki_414beed608bf11ebbd8aac1f6bf848b6,"Which film has the director who died first, Wh...",bridge_comparison,bridge_comparison,single,1,0,0,5,medium,7,,,
2,2wiki_414beed408bf11ebbd8aac1f6bf848b6,Which film has the director who is older than ...,bridge_comparison,bridge_comparison,single,1,0,0,5,medium,7,,,
3,2wiki_9e841e45087611ebbd67ac1f6bf848b6,"Which film whose director is younger, Time To ...",bridge_comparison,bridge_comparison,single,1,0,0,4,medium,6,,,
4,2wiki_1650b5a6086111ebbd5dac1f6bf848b6,Which film has the director who was born earli...,bridge_comparison,bridge_comparison,single,1,0,0,4,medium,6,,,
5,2wiki_2a221b8808d411ebbd96ac1f6bf848b6,Which film has the director who was born earli...,bridge_comparison,bridge_comparison,single,1,0,0,4,medium,6,,,
6,2wiki_023b608f087a11ebbd67ac1f6bf848b6,"Which film has the director who died first, Da...",bridge_comparison,bridge_comparison,single,1,0,0,4,medium,6,,,
7,2wiki_c8d52ef108c011ebbd8aac1f6bf848b6,Which film has the director who was born earli...,bridge_comparison,bridge_comparison,single,1,0,0,4,medium,6,,,
8,2wiki_bfb6685c088911ebbd6eac1f6bf848b6,"Which film has the director who is older, In T...",bridge_comparison,bridge_comparison,single,1,0,0,4,medium,6,,,
9,2wiki_64e077b608b011ebbd84ac1f6bf848b6,Which film has the director who is older than ...,bridge_comparison,bridge_comparison,single,1,0,0,4,medium,6,,,


In [14]:
# =========================
# ЭКСПОРТ JSONL / JSON / CSV
# =========================
all_records = df.to_dict(orient="records")
cand_records = cand_df.to_dict(orient="records")
translation_minimal_records = translation_minimal_df.to_dict(orient="records")

write_jsonl(all_records, JSONL_DIR / "2wikimultihop_all_normalized.jsonl")
write_json(all_records, JSONL_DIR / "2wikimultihop_all_normalized.json")
df.to_csv(CSV_DIR / "2wikimultihop_all_normalized.csv", index=False)

write_jsonl(cand_records, JSONL_DIR / "2wikimultihop_strong_multihop_candidates.jsonl")
write_json(cand_records, JSONL_DIR / "2wikimultihop_strong_multihop_candidates.json")
cand_df.to_csv(CSV_DIR / "2wikimultihop_strong_multihop_candidates.csv", index=False)

write_jsonl(translation_minimal_records, JSONL_DIR / "2wikimultihop_translation_minimal.jsonl")
write_json(translation_minimal_records, JSONL_DIR / "2wikimultihop_translation_minimal.json")
translation_minimal_df.to_csv(CSV_DIR / "2wikimultihop_translation_minimal.csv", index=False)

manual_review_df.to_csv(CSV_DIR / "2wikimultihop_manual_review_sheet.csv", index=False)

write_json(summary, REPORTS_DIR / "summary_stage1.json")

print("[saved]")
for p in [
    JSONL_DIR / "2wikimultihop_all_normalized.jsonl",
    JSONL_DIR / "2wikimultihop_strong_multihop_candidates.jsonl",
    JSONL_DIR / "2wikimultihop_translation_minimal.json",
    CSV_DIR / "2wikimultihop_manual_review_sheet.csv",
    REPORTS_DIR / "summary_stage1.json",
]:
    print(" -", p)

[saved]
 - /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1/jsonl/2wikimultihop_all_normalized.jsonl
 - /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1/jsonl/2wikimultihop_strong_multihop_candidates.jsonl
 - /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1/jsonl/2wikimultihop_translation_minimal.json
 - /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1/csv/2wikimultihop_manual_review_sheet.csv
 - /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1/reports/summary_stage1.json


In [15]:
# =========================
# PREVIEW
# =========================
display_cols = [
    "benchmark_id",
    "question_en_original",
    "question_type",
    "answer_mode",
    "num_answers",
    "num_evidences",
    "num_supporting_facts",
    "shallow_bridge_flag",
    "strength_bucket",
    "reconstructability_score",
]
cand_df[display_cols].head(30)

,benchmark_id,question_en_original,question_type,answer_mode,num_answers,num_evidences,num_supporting_facts,shallow_bridge_flag,strength_bucket,reconstructability_score
0,2wiki_e07552fc08a111ebbd7aac1f6bf848b6,"Which film has the director who is older, The ...",bridge_comparison,single,1,0,0,False,medium,7
1,2wiki_414beed608bf11ebbd8aac1f6bf848b6,"Which film has the director who died first, Wh...",bridge_comparison,single,1,0,0,False,medium,7
2,2wiki_414beed408bf11ebbd8aac1f6bf848b6,Which film has the director who is older than ...,bridge_comparison,single,1,0,0,False,medium,7
3,2wiki_9e841e45087611ebbd67ac1f6bf848b6,"Which film whose director is younger, Time To ...",bridge_comparison,single,1,0,0,False,medium,6
4,2wiki_1650b5a6086111ebbd5dac1f6bf848b6,Which film has the director who was born earli...,bridge_comparison,single,1,0,0,False,medium,6
5,2wiki_2a221b8808d411ebbd96ac1f6bf848b6,Which film has the director who was born earli...,bridge_comparison,single,1,0,0,False,medium,6
6,2wiki_023b608f087a11ebbd67ac1f6bf848b6,"Which film has the director who died first, Da...",bridge_comparison,single,1,0,0,False,medium,6
7,2wiki_c8d52ef108c011ebbd8aac1f6bf848b6,Which film has the director who was born earli...,bridge_comparison,single,1,0,0,False,medium,6
8,2wiki_bfb6685c088911ebbd6eac1f6bf848b6,"Which film has the director who is older, In T...",bridge_comparison,single,1,0,0,False,medium,6
9,2wiki_64e077b608b011ebbd84ac1f6bf848b6,Which film has the director who is older than ...,bridge_comparison,single,1,0,0,False,medium,6
